In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os

os.environ['ENV_TYPE'] = 'swe-bench'
from docent._loader.load_inspect import load_swebench

TRANSCRIPTS = load_swebench()

In [ ]:
len(TRANSCRIPTS)

In [ ]:
from docent.data_models.agent_run import AgentRun
from typing import cast

def find_by_expid(transcripts: list[AgentRun], expid: str) -> AgentRun:
    for t in transcripts:
        if t.metadata.get("experiment_id") == expid:
            return t
    raise ValueError(f"No transcript found for experiment ID {expid}")

def printw(s: str):
    LINE_LENGTH = 190
    lines = s.split('\n')
    for line in lines:
        for i in range(0, len(line), LINE_LENGTH):
            print(line[i:i+LINE_LENGTH])

In [ ]:

all_sample_ids: set[str] = {
    "astropy__astropy-14096", "django__django-11551", "django__django-11239", "django__django-14351",
    "django__django-11299", "django__django-11749", "django__django-12193", "django__django-14404",
}

TRANSCRIPT_PAIRS: dict[str, list[AgentRun]] = {}
for sample_id in all_sample_ids:
    sample_transcripts = [t for t in TRANSCRIPTS if t.metadata.get("sample_id") == sample_id]
    if len(sample_transcripts) != 2:
        continue
    first, second = find_by_expid(sample_transcripts, 'swebench-sonnet37-tools'), find_by_expid(sample_transcripts, 'swebench-sonnet35-tools')
    TRANSCRIPT_PAIRS[sample_id] = [first, second]

len(TRANSCRIPT_PAIRS)

In [ ]:
import os
os.environ['ENV_TYPE'] = 'o3'


from docent._loader.load_o3 import load_o3

O3_TRANSCRIPTS = load_o3()

o3_all_sample_ids: set[str] = set([t.metadata.get("sample_id") for t in O3_TRANSCRIPTS])

print(o3_all_sample_ids)
O3_TRANSCRIPT_PAIRS: dict[str, list[AgentRun]] = {}
for sample_id in o3_all_sample_ids:
    sample_transcripts = [t for t in O3_TRANSCRIPTS if t.metadata.get("sample_id") == sample_id]
    if len(sample_transcripts) != 2:
        continue
    first, second = find_by_expid(sample_transcripts, 'human-generated_attacks'), find_by_expid(sample_transcripts, 'human-generated_attacks_2')
    O3_TRANSCRIPT_PAIRS[sample_id] = [first, second]

len(O3_TRANSCRIPT_PAIRS)

In [ ]:
from docent._loader.load_inspect import load_agentharm

AGENTHARM_TRANSCRIPTS = load_agentharm()

agentharm_all_sample_ids: set[str] = set([t.metadata.get("sample_id") for t in AGENTHARM_TRANSCRIPTS])

AGENTHARM_TRANSCRIPT_PAIRS: dict[str, list[AgentRun]] = {}
for sample_id in agentharm_all_sample_ids:
    sample_transcripts = [t for t in AGENTHARM_TRANSCRIPTS if t.metadata.get("sample_id") == sample_id]
    if len(sample_transcripts) != 2:
        continue
    first, second = find_by_expid(sample_transcripts, 'gpt4o-default'), find_by_expid(sample_transcripts, 'gpt4o-react')
    AGENTHARM_TRANSCRIPT_PAIRS[sample_id] = [first, second]

len(AGENTHARM_TRANSCRIPT_PAIRS)

In [ ]:
from docent.data_models.transcript import (
    MULTI_BLOCK_CITE_INSTRUCTION,
    SINGLE_BLOCK_CITE_INSTRUCTION,
)
from docent._llm_util.prod_llms import get_llm_completions_async
from docent._llm_util.providers.preference_types import PROVIDER_PREFERENCES

# Initial Attempts

In [ ]:
from docent._ai_tools.search import execute_search

res = await execute_search(TRANSCRIPTS[0:16], "times the agent did smth weird")

# Multi-Stage

In [ ]:
# state reasoning to preprocess

from docent._llm_util.data_models.llm_output import LLMOutput

async def extract_states(
    transcript: AgentRun,
) -> str:
    prompt = f"""
Here is a sequence of actions an agent took to solve a task.
{transcript.transcripts["default"].to_str()}

Note that each message in the sequence can have one of several roles - system, user, assistant, or tool.

For each ASSISTANT message, perform the following procedure:
- Summarize the action taken in the message
- Summarize the goal of the agent's current action
- Provide a concise but specific summary of the agent's past actions that are relevant to the current goal. You are encouraged to cite evidence from the transcripts: {SINGLE_BLOCK_CITE_INSTRUCTION}

Do not mention non-assistant messages in your output.

Here are some examples of the level of specifity in which we'd like to describe the messages in.

Action: The agent uses grep to find the test.py file.
Goal: The agent is trying to explore the codebase and read code relevant to the task.capitalize
Relevant past actions: None, this is the first action taken by the agent.

Action: The agent is editing OAuth configuration settings in its test script.
Goal: The agent is trying to get its test script to run without errors.
Relevant past actions: The agent previously wrote a test script, but upon execution it produced an OutOfBoundsError. <further explanation and citations>

Action: The agent is writing a detailed test script that tests for many edge cases.
Goal: The agent is trying to test its solution to ensure correctness.
Relevant past actions: The agent previously implemented a solution that resolves the issue by modifying the function to use sets instead of lists. <further explanation and citations>

Format your output as follows:
[B<message_idx>]
Action: [action taken]
Goal: [goal of the action]
Relevant past actions: [summary of past actions that are relevant to the current goal, with citations]
[B<message_idx>]
Action: [action taken]
Goal: [goal of the action]
Relevant past actions: [summary of past actions that are relevant to the current goal, with citations]
...
    """.strip()

    result = ""

    async def _streaming_callback(batch_index: int, llm_output: LLMOutput):
        nonlocal result

        result = llm_output.completions[0].text


    outputs = await get_llm_completions_async(
        [
            [
                {
                    "role": "user",
                    "content": prompt,
                },
            ]
        ],
        PROVIDER_PREFERENCES.compare_transcripts,
        max_new_tokens=8192 * 4,
        timeout=240.0,
        use_cache=True,
        streaming_callback=_streaming_callback,
    )

    text = outputs[0].first_text
    if text is None:
        return ""
    return text

class MessageState:
    def __init__(self, message_idx: int, action: str, goal: str, past_actions: str):
        self.message_idx = message_idx
        self.action = action
        self.goal = goal
        self.past_actions = past_actions
    
    def __str__(self):
        return f"[B{self.message_idx}]\nAction: {self.action}\nGoal: {self.goal}\nRelevant past actions: {self.past_actions}"

def parse_output(output: str) -> list[MessageState]:
    result: list[MessageState] = []
    idx, action, goal, past_actions = -1, "", "", ""
    for line in output.split('\n'):
        if line.startswith('[B'):
            idx = int(line.removeprefix('[B').removesuffix(']'))
            action, goal, past_actions = "", "", ""
        elif line.startswith('Action:'):
            action = line.removeprefix('Action:').strip()
        elif line.startswith('Goal:'):
            goal = line.removeprefix('Goal:').strip()
        elif line.startswith('Relevant past actions:'):
            past_actions = line.removeprefix('Relevant past actions:').strip()
            result.append(MessageState(idx, action, goal, past_actions))
    return result

In [ ]:
# single pass, inline state reasoning with better instructions
from docent.data_models.chat.message import AssistantMessage

def format_transcript_messages_and_states(transcript: AgentRun, states: list[MessageState], transcript_idx_label: int):
    result = ""
    for state in states:
        message = transcript.transcripts["default"].messages[state.message_idx]
        if not isinstance(message, AssistantMessage):
            continue
        result += f"[T{transcript_idx_label}B{state.message_idx}]\n"
        result += f"Goal: {state.goal}\n"
        result += f"Relevant past actions: {state.past_actions}\n"
        result += f"Action: {state.action}\n"
        result += "Raw message: " + transcript.transcripts["default"].messages[state.message_idx].text + '\n'
        result += '-' * 32 + '\n'
    return result

async def compare_transcripts(
    transcript_1: AgentRun,
    transcript_2: AgentRun,
    states_1: list[MessageState],
    states_2: list[MessageState],
) -> str:
    prompt = f"""
Here are two different sequences of actions an agent took to solve a task. For each transcript, you will be given messages in the following format:

<message_idx_label>
Goal: [goal of the current action]
Relevant past actions: [summary of past actions that are relevant to the current goal, with citations]
Action: [a summary of the action taken]
Raw message: [raw message containing the action]
--------------------------------

First transcript:
{format_transcript_messages_and_states(transcript_1, states_1, 0)}
Second transcript:
{format_transcript_messages_and_states(transcript_2, states_2, 1)}

We care about instances where the two agents take different actions.
We are especially interested in cases where two agents have the same goal and context but take different actions (because that implies the difference in actions stems solely from a difference in the agents themselves), in which case you should mention the shared goals and context explicitly.
Cases where the two agents have different goals or context and take different actions may also be relevant, if the difference in actions is very interesting. In this case, you should mention the shared context/goals between the two agents, and also point out differences in the context/goals that may have caused the difference in observed actions.
Note that the ground truth is always the raw message, which shows the action the agent actually took; the summaries are provided to give you context, but we only wish to surface differences in the actual agent behaviors (rather than differences in provided summaries).

Here are some examples of differences, and the level of specifity in which we'd like them to be described:
<claim>
Both agents have read the task description and are trying to locate the test.py file, but agent 1 uses grep and succeeds while agent 2 uses tool calls and fails
</claim>
<claim>
Both models are trying to test their solutions, and agent 1 writes much more detailed tests than agent 2. However, this may partially be explained by the fact that agent 1's solution is more complex than agent 2's solution.
</claim>

Look through the transcripts and list the major differences in actions between the two agents. If there are no major differences, minor differences are also fine.

Use these guidelines for citations: {MULTI_BLOCK_CITE_INSTRUCTION}

Format your final list as follows:
<claim>
both agents have similar context W and want to accomplish X, but agent 1 does Y while agent 2 does Z
</claim>
<evidence>
explain W, X, Y, Z, with citations
</evidence>
<claim>
both agents want to accomplish X, but agent 1 does Y while agent 2 does Z. however, this difference may be partially explained by agent 1 having context W, while agent 2 has context W'
</claim>
<evidence>
explain X, Y, Z, W, W', with citations
</evidence>
...

Do not respond with any other text than the list of claims and evidence.
Always refer to the first transcript as "Agent 1" and the second as "Agent 2".
Explicitly mention the different actions each agents took. Explicitly qualify claims by stating which context and goals are shared between the agents, and which are different (and how they are different).
    """.strip()

    result = ""

    async def _streaming_callback(batch_index: int, llm_output: LLMOutput):
        nonlocal result

        result = llm_output.completions[0].text

    outputs = await get_llm_completions_async(
        [
            [
                {
                    "role": "user",
                    "content": prompt,
                },
            ]
        ],
        PROVIDER_PREFERENCES.compare_transcripts[0:1],
        max_new_tokens=8192 * 5,
        timeout=240.0,
        use_cache=True,
        streaming_callback=_streaming_callback,
    )

    text = outputs[0].first_text
    if text is None:
        return ""
    return text

In [ ]:
# from tqdm.asyncio import tqdm
# from IPython.display import clear_output


# async def extract_states_and_diffs(transcript_pairs: dict[str, list[Transcript]]):

#     states: dict[tuple[str, int], str] = {}
#     all_sample_ids = list(transcript_pairs.keys())
#     tasks = [
#         extract_states(t)
#         for sample_id in all_sample_ids for t in transcript_pairs[sample_id]
#     ]
#     results = await tqdm.gather(*tasks)
#     for i, result in enumerate(results):
#         states[(all_sample_ids[i//2], i%2)] = result

#     diffs: dict[str, str] = {}
#     tasks = [
#         compare_transcripts_5(
#             transcript_pairs[sample_id][0], transcript_pairs[sample_id][1],
#             parse_output(states[(sample_id, 0)]), parse_output(states[(sample_id, 1)])
#         )
#         for sample_id in all_sample_ids
#     ]
#     results = await tqdm.gather(*tasks)
#     for i, result in enumerate(results):
#         diffs[all_sample_ids[i]] = result
#     clear_output()
#     for k in states:
#         print(k, transcript_pairs[k[0]][k[1]].metadata.scores)
#         for line in (parse_output(states[k])):
#             print(line)
#         print('-' * 100)
#     for k in diffs:
#         print(k, transcript_pairs[k][0].metadata.scores, transcript_pairs[k][1].metadata.scores)
#         printw(diffs[k])
#         print('-' * 100)

In [ ]:
from tqdm.asyncio import tqdm
from IPython.display import clear_output

def parse_diff_output(output: str) -> list[tuple[str, str]]:
    result: list[tuple[str, str]] = []
    curr_index: int = 0
    # get lines between <claim> and </claim>
    while True:
        start_claim_index = output.find('<claim>', curr_index)
        if start_claim_index == -1:
            break
        end_claim_index = output.find('</claim>', start_claim_index)
        if end_claim_index == -1:
            break
        start_evidence_index = output.find('<evidence>', start_claim_index)
        if start_evidence_index == -1:
            break
        end_evidence_index = output.find('</evidence>', start_evidence_index)
        if end_evidence_index == -1:
            break
        result.append((output[start_claim_index + len('<claim>'):end_claim_index].strip(), output[start_evidence_index + len('<evidence>'):end_evidence_index].strip()))
        curr_index = end_evidence_index + 1
    return result

async def extract_states_and_diffs(transcript_pairs: dict[str, list[AgentRun]]) -> dict[str, str]:

    states: dict[tuple[str, int], str] = {}
    all_sample_ids = list(transcript_pairs.keys())
    tasks = [
        extract_states(t)
        for sample_id in all_sample_ids for t in transcript_pairs[sample_id]
    ]
    results = await tqdm.gather(*tasks)
    for i, result in enumerate(results):
        states[(all_sample_ids[i//2], i%2)] = result

    diffs: dict[str, str] = {}
    tasks = [
        compare_transcripts(
            transcript_pairs[sample_id][0], transcript_pairs[sample_id][1],
            parse_output(states[(sample_id, 0)]), parse_output(states[(sample_id, 1)])
        )
        for sample_id in all_sample_ids
    ]
    results = await tqdm.gather(*tasks)
    for i, result in enumerate(results):
        diffs[all_sample_ids[i]] = result
    clear_output()
    for k in states:
        print(k, transcript_pairs[k[0]][k[1]].metadata.scores)
        for line in (parse_output(states[k])):
            print(line)
        print('-' * 100)
    for k in diffs:
        print(k, transcript_pairs[k][0].metadata.scores, transcript_pairs[k][1].metadata.scores)
        for diff in parse_diff_output(diffs[k]):
            print(f"Claim: {diff[0]}")
            print(f"Evidence: {diff[1]}")
        print('-' * 100)
    return diffs

In [ ]:
swe_diffs = await extract_states_and_diffs(TRANSCRIPT_PAIRS)

In [ ]:
o3_diffs = await extract_states_and_diffs(O3_TRANSCRIPT_PAIRS)

In [ ]:
agentharm_diffs = await extract_states_and_diffs(AGENTHARM_TRANSCRIPT_PAIRS)

# Aggregation

In [ ]:
#from docent._frames.diff import compare_transcripts


In [ ]:
#from docent._frames.diff import compute_diff_and_evidence

In [ ]:
def extract_themes(llm_output: str | None) -> list[str]:
    if llm_output is None:
        return []
    results: list[str] = []
    start_index = llm_output.find('<theme_start>')
    index = 1
    while start_index != -1:
        end_index = llm_output.find('<theme_end>', start_index)
        substring = llm_output[start_index:end_index]
        substring = substring.removeprefix('<theme_start>')
        substring = substring.removeprefix("\n")
        substring = substring.removeprefix(f"Theme {index}:")
        results.append(substring.strip())
        start_index = llm_output.find('<theme_start>', end_index)
        index += 1
    return results

def format_diffs(diffs: dict[str, str]) -> str:
    result = ""
    for diff in diffs.values():
        result += f"{diff}\n"
    return result

# async def aggregate_differences(diffs: dict[str, str]):
#     prompt = f"""You will be given a list of summaries of differences between two agents' performances on a variety of tasks. The list will be given in the following format:

# <claim>
# Agent 1 and agent 2 were both trying to accomplish X, but agent 1 did Y while agent 2 did Z.
# </claim>
# <evidence>
# evidence for the claim, eg. specific examples where agent 1 is more of X than agent 2
# </evidence>
# ----------------
# <claim>
# Agent 1 and agent 2 were both trying to accomplish X', but agent 1 did Y' while agent 2 did Z'.
# </claim>
# <evidence>
# evidence for the claim, eg. specific examples where agent 1 is more of X' than agent 2
# </evidence>
# ----------------
# ...

# Based on this list, please propose a list of recurring themes where the first agent and the second agent consistently have different behaviors. Avoid repeating yourself in the output.
# Try to choose recurring themes where the evidence for the theme clearly outweighs evidence in the reverse direction.

# Format your output in this format:

# <theme_start>
# Description of the theme and how it relates to agent 1 and agent 2
# <theme_end>
# <theme_start>
# Description of the theme and how it relates to agent 1 and agent 2
# <theme_end>
# ...

# Here is the list of differences:

# {format_diffs(diffs)}
#     """.strip()
#     outputs = await get_llm_completions_async(
#         [
#             [
#                 {
#                     "role": "user",
#                     "content": prompt,
#                 },
#             ]
#         ],
#         PROVIDER_PREFERENCES.compare_transcripts,
#         max_new_tokens=8192 * 2,
#         timeout=180.0,
#         use_cache=True,
#     )

#     return extract_themes(outputs[0].completions[0].text)




In [ ]:
from docent._ai_tools.clustering.cluster_assigner import LlmApiClusterAssigner

def split_diff_into_claims(diff: str) -> list[tuple[str, str]]:
    result: list[tuple[str, str]] = []
    curr_index = 0
    # get lines between <claim> and </claim>
    start_claim_index = diff.find('<claim>', curr_index)
    end_claim_index = diff.find('</claim>', curr_index)
    start_evidence_index = diff.find('<evidence>', curr_index)
    end_evidence_index = diff.find('</evidence>', curr_index)
    while start_claim_index != -1 and end_claim_index != -1 and start_evidence_index != -1 and end_evidence_index != -1:
        result.append((diff[start_claim_index + len('<claim>'):end_claim_index].strip(), diff[start_evidence_index + len('<evidence>'):end_evidence_index].strip()))
        curr_index = end_evidence_index + 1
        start_claim_index = diff.find('<claim>', curr_index)
        end_claim_index = diff.find('</claim>', curr_index)
        start_evidence_index = diff.find('<evidence>', curr_index)
        end_evidence_index = diff.find('</evidence>', curr_index)
    return result


def extract_fn(llm_output: str) -> list[str]:
    results: list[str] = []
    start_index = llm_output.find('<theme_start>')
    index = 1
    while start_index != -1:
        end_index = llm_output.find('<theme_end>', start_index)
        substring = llm_output[start_index:end_index]
        substring = substring.removeprefix('<theme_start>')
        substring = substring.removeprefix("\n")
        substring = substring.removeprefix(f"Theme {index}:")
        results.append(substring.strip())
        start_index = llm_output.find('<theme_start>', end_index)
        index += 1
    return results

def prompt_build_fn(extra_instructions: str, diffs: list[str]) -> str:
    prompt = f"""You will be given a list of summaries of differences between two agents' performances on a variety of tasks. The list will be given in the following format:

<claim>
Agent 1 and agent 2 were both trying to accomplish X, but agent 1 did Y while agent 2 did Z.
</claim>
<evidence>
evidence for the claim, eg. specific examples where agent 1 is more of X than agent 2
</evidence>
----------------
<claim>
Agent 1 and agent 2 were both trying to accomplish X', but agent 1 did Y' while agent 2 did Z'.
</claim>
<evidence>
evidence for the claim, eg. specific examples where agent 1 is more of X' than agent 2
</evidence>
----------------
...

Based on this list, please propose a list of recurring themes where the first agent and the second agent consistently have different behaviors. Avoid repeating yourself in the output.
Try to choose recurring themes where the evidence for the theme clearly outweighs evidence in the reverse direction.

Themes should contain exactly one idea/concept each.
Themes should be mutually exclusive: no two themes should describe the same thing.
Themes should be collectively exhaustive: no item of the list should be left out.

Format your output in this format:

<theme_start>
Description of the theme and how it relates to agent 1 and agent 2
<theme_end>
<theme_start>
Description of the theme and how it relates to agent 1 and agent 2
<theme_end>
...

{extra_instructions}

Here is the list of differences:

{'\n----------------\n'.join(diffs)}
    """.strip()
    return prompt

from docent._ai_tools.clustering.multi_round_clustering import Cluster, cluster_from_initial_proposal
    
def assign_prompt_fn(item: str, cluster: str) -> str:
    ASSIGNMENT_PROMPT = """
You are given a claim C and a behavior B.
Your task is to determine whether B is a direct example of C.

The claim may come with examples, which you shouldn't treat as strict requirements.

Return two lines in the following exact format:
- ANSWER: <YES/NO>
- EXPLANATION: <concise but specific explanation; no more than a few high-information words>

Only reply yes if B is a direct example of C; if B and C are correlated but distinct behaviors, this does not count.

Here is your input:
C: {cluster}
B: {item}
""".strip()
    stripped_item = item.removeprefix("<claim>").split("</claim>")[0].strip()
    return ASSIGNMENT_PROMPT.format(cluster=cluster, item=stripped_item)

assigner = LlmApiClusterAssigner.from_sonnet_37_thinking(assign_prompt_fn)

async def prune_fn(clusters: list[Cluster], attributes: list[str], assigner: LlmApiClusterAssigner) -> list[Cluster]:
    cluster_list: list[Cluster] = []
    for cluster in clusters:
        assert isinstance(cluster, Cluster), clusters
        centroid = cluster.centroid
        reverse_centroid = centroid.replace("Agent 1", "Agent 3").replace("Agent 2", "Agent 1").replace("Agent 3", "Agent 2")
        results = await assigner.assign(attributes + attributes, [centroid, ] * len(attributes) + [reverse_centroid, ] * len(attributes))
        first_half_matches = [i for i, r in enumerate(results[:len(attributes)]) if r is not None and r[0]]
        second_half_matches = [i for i, r in enumerate(results[len(attributes):]) if r is not None and r[0]]
        if len(first_half_matches) > max(1.2 * len(second_half_matches), len(second_half_matches) + 1):
            cluster.metadata['reverse_indices'] = second_half_matches
            cluster_list.append(cluster)
    return cluster_list


In [ ]:
all_swe_diffs: list[str] = []
for v in swe_diffs.values():
    curr_items = split_diff_into_claims(v)
    all_swe_diffs.extend([f"""<claim>
{item[0]}
</claim>
<evidence>
{item[1]}
</evidence>""" for item in curr_items])

swe_clusters = await cluster_from_initial_proposal(
    all_swe_diffs,
    "ways in which agent 1 and agent 2 differ",
    assigner,
    prune_fn=prune_fn,
    clustering_prompt_fn=prompt_build_fn,
    output_extractor=extract_fn,
)

clear_output()

for cluster in swe_clusters:
    print(cluster.indices)
    printw(cluster.centroid)
    print(cluster.metadata)
    print('-' * 100)

# swe_themes_list = await aggregate_differences(swe_diffs)

# for theme in swe_themes_list:
#     print(theme)
#     print('-' * 100)

In [ ]:
all_o3_diffs: list[str] = []
for v in o3_diffs.values():
    curr_items = split_diff_into_claims(v)
    all_o3_diffs.extend([f"""<claim>
{item[0]}
</claim>
<evidence>
{item[1]}
</evidence>""" for item in curr_items])

o3_clusters = await cluster_from_initial_proposal(
    all_o3_diffs,
    "ways in which agent 1 and agent 2 differ",
    assigner,
    prune_fn=prune_fn,
    clustering_prompt_fn=prompt_build_fn,
    output_extractor=extract_fn,
)

clear_output()

for cluster in o3_clusters:
    print(cluster.indices)
    printw(cluster.centroid)
    print(cluster.metadata)
    print('-' * 100)

# swe_themes_list = await aggregate_differences(swe_diffs)

# for theme in swe_themes_list:
#     print(theme)
#     print('-' * 100)

In [ ]:
all_agentharm_diffs: list[str] = []
index_to_transcript = {}
counter = 0
for k, v in agentharm_diffs.items():
    curr_items = split_diff_into_claims(v)
    all_agentharm_diffs.extend([f"""<claim>
{item[0]}
</claim>
<evidence>
{item[1]}
</evidence>""" for item in curr_items])
    for item in curr_items:
        index_to_transcript[counter] = k
        counter += 1

agentharm_clusters = await cluster_from_initial_proposal(
    all_agentharm_diffs,
    "ways in which agent 1 and agent 2 differ",
    assigner,
    prune_fn=prune_fn,
    clustering_prompt_fn=prompt_build_fn,
    output_extractor=extract_fn,
)
clear_output()
ordering = [1, 0, 2, 3, 4, 7, 5, 6]
for i in ordering:
    cluster = agentharm_clusters[i]
    print(f"Transcripts supporting: {[index_to_transcript[i] for i in cluster.indices]}")
    print(f"Transcripts opposing: {[index_to_transcript[i] for i in cluster.metadata['reverse_indices']]}")
    printw(cluster.centroid)
    print('-' * 100)

# Old aggregations

In [ ]:

#assigner.temperature = 1.0

async def evaluate_new_queries(
    diffs: dict[str, str], diff_clusters: list[str]
) -> dict[str, list[list[tuple[str, int]]]]:
    items: list[str] = []
    clusters: list[str] = []
    all_diff_items: list[tuple[str, str, int]] = []
    full_diff_clusters: list[str] = []
    full_diff_clusters.extend(diff_clusters)
    for cluster in diff_clusters:
        new_cluster = cluster.replace("Agent 1", "Agent 3").replace("Agent 2", "Agent 1").replace("Agent 3", "Agent 2")
        full_diff_clusters.append(new_cluster)
    for diff_id, diff in diffs.items():
        for i, (claim, _evidence) in enumerate(split_diff_into_claims(diff)):
            #all_diff_items.append((f"Claim: {claim}\n Evidence: {evidence}", diff_id, i))
            all_diff_items.append((f"Claim: {claim}", diff_id, i))
    for claim, _, _ in all_diff_items:
        items.extend(
            [
                claim,
            ] * len(full_diff_clusters)
        )
        clusters.extend(full_diff_clusters)
    results = await assigner.assign(items, clusters)
    final_results: dict[str, list[list[tuple[str, int]]]] = {}
    for i, cluster in enumerate(diff_clusters):
        final_results[cluster] = [[], []]
        for j in range(i, len(results), 2 * len(diff_clusters)):
            if results[j] is None:
                continue
            if cast(tuple[bool, str], results[j])[0]:
                final_results[cluster][0].append(all_diff_items[j // len(full_diff_clusters)][1:])
        for j in range(i + len(diff_clusters), len(results), 2 * len(diff_clusters)):
            if results[j] is None:
                continue
            if cast(tuple[bool, str], results[j])[0]:
                final_results[cluster][1].append(all_diff_items[j // len(full_diff_clusters)][1:])
    return final_results


In [ ]:
contract_citations = await evaluate_new_queries(contract_diffs, contract_themes_list)

clear_output()

for cluster in contract_citations:
    printw(cluster)
    print(f"Occurs in {sum(len(x) for x in contract_citations[cluster][0])} of transcripts")
    print(f"Opposite occurs in {sum(len(x) for x in contract_citations[cluster][1])} of transcripts")
    print('=' * 100)


In [ ]:
o3_citations = await evaluate_new_queries(o3_diffs, o3_themes_list)

clear_output()

for cluster in o3_citations:
    printw(cluster)
    print(f"Occurs in {sum(len(x) for x in o3_citations[cluster][0])} of transcripts")
    print(f"Opposite occurs in {sum(len(x) for x in o3_citations[cluster][1])} of transcripts")
    # for diff_id, diff_idx in o3_citations[cluster][0]:
    #     print('-' * 100)
    #     claim, evidence = (split_diff_into_claims(o3_diffs[diff_id])[diff_idx])
    #     printw(f"Claim: {claim}\n Evidence: {evidence}")
    # print('=' * 100)
    # for diff_id, diff_idx in o3_citations[cluster][1]:
    #     print('-' * 100)
    #     claim, evidence = (split_diff_into_claims(o3_diffs[diff_id])[diff_idx])
    #     printw(f"Reverse Claim: {claim}\n Reverse Evidence: {evidence}")
    print('=' * 100)


In [ ]:
swe_citations = await evaluate_new_queries(swe_diffs, swe_themes_list)

clear_output()

for cluster in swe_citations:
    printw(cluster)
    print(f"Occurs in {sum(len(x) for x in swe_citations[cluster][0])} of transcripts")
    print(f"Opposite occurs in {sum(len(x) for x in swe_citations[cluster][1])} of transcripts")
    for diff_id, diff_idx in swe_citations[cluster][0]:
        print('-' * 100)
        claim, evidence = (split_diff_into_claims(swe_diffs[diff_id])[diff_idx])
        printw(f"Claim: {claim}\n Evidence: {evidence}")
    print('=' * 100)
    for diff_id, diff_idx in swe_citations[cluster][1]:
        print('-' * 100)
        claim, evidence = (split_diff_into_claims(swe_diffs[diff_id])[diff_idx])
        printw(f"Reverse Claim: {claim}\n Reverse Evidence: {evidence}")
    print('=' * 100)


In [ ]:
[87, 37, 75, 25, 75, 37, 62, 0, 75, 12]
[87, 37, 87, 25, 62, 25, 62, 12, 87, 12]
[87, 37, 75, 25, 87, 25, 62, 12, 75, 25]
[87, 37, 75, 25, 50, 37, 62, 0, 87, 12]

[50, 25, 50, 0, 50, 0, 37, 0, 62, 0]

# O3

In [ ]:
2+3

# Loading transcripts

In [ ]:
import os

os.environ['ENV_TYPE'] = 'swe-bench'
from docent._loader.load_inspect import load_swebench

TRANSCRIPTS = load_swebench()

len(TRANSCRIPTS)

In [ ]:
from docent import DocentClient

from docent._env_util.env import ENV


client = DocentClient(server_url="http://localhost:8889", web_url="http://localhost:3001", email="vincent@transluce.org")

# collection_id = client.create_collection(name="AGENTHARM", description="example that comes with the Docent repo")
collection_id = "abc44fe8-d799-4c1e-949a-32debd1dfd72"
collection_id

In [ ]:
from docent._loader.load_inspect import load_agentharm

AGENTHARM_TRANSCRIPTS = load_agentharm()
client.add_agent_runs(collection_id, AGENTHARM_TRANSCRIPTS)

In [ ]:
# res = await db.compute_diff_clusters(ctx=ViewContext(
#     collection_id=collection_id,
#     view_id="",
#     base_filter=None
# ), experiment_id_1="gpt4o-default", experiment_id_2="gpt4o-react")

res = await db.compute_diff_search(
    ctx=ViewContext(
    collection_id=collection_id,
    view_id="",
    base_filter=None
),
    experiment_id_1="gpt4o-default",
    experiment_id_2="gpt4o-react",
    # search_query="Agent 2 offers follow-up assistance",
    search_query="Agent 1 provides info before refusing to assist, while Agent 2 responds minimally"

    #Agent 1 provides more detailed information about potentially harmful topics before refusing to assist, while Agent 2 offers minimal engagement with ha

)

from IPython.display import clear_output

clear_output()

for x in res:
    if x[1] != 0:
        print(x[1], x[0])


In [ ]:

for x in (res):
    printw(x)

In [ ]:
from docent import DocentClient

from docent._env_util.env import ENV

client = DocentClient(server_url="http://localhost:8889", web_url="http://localhost:3001", email="vincent@transluce.org")

collection_id = client.create_collection(name="SWEBENCH", description="example that comes with the Docent repo")
collection_id = "e292429e-5f85-4d91-925f-20a34398e3f3"
collection_id

In [ ]:
SWE_TRANSCRIPTS = load_swebench()
client.add_agent_runs(collection_id, SWE_TRANSCRIPTS)

In [ ]:
# from docent.data_models import AgentRun, Transcript

# isinstance(TRANSCRIPTS[0], Transcript)
# isinstance(TRANSCRIPTS[0], AgentRun)

# client.add_agent_runs(collection_id, TRANSCRIPTS)

In [ ]:
from docent._db_service.service import DBService
db = await DBService.init()

In [ ]:
# res = await db.compute_diffs(collection_id, "swebench-sonnet37-tools", "swebench-sonnet35-tools")
from docent._db_service.contexts import ViewContext

# res = await db.compute_diff_clusters(ctx=ViewContext(
#     collection_id=collection_id,
#     view_id="",
#     base_filter=None
# ), experiment_id_1="swebench-sonnet37-tools", experiment_id_2="swebench-sonnet35-tools")

res = await db.compute_diff_search(
    ctx=ViewContext(
    collection_id=collection_id,
    view_id="",
    base_filter=None
),
    experiment_id_1="swebench-sonnet37-tools",
    experiment_id_2="swebench-sonnet35-tools",
    search_query="Agent 1 writes better tests than Agent 2",
)

from IPython.display import clear_output

clear_output()


In [ ]:
[x for x in res if x[1] != 0][:5]

In [ ]:
res

# Archive

In [ ]:
# single pass, inline state reasoning

from docent._llm_util.types import LLMOutput

async def compare_transcripts_3(
    transcript_1: Transcript,
    transcript_2: Transcript,
) -> str:
    prompt = f"""
Here are two different sequences of actions an agent took to solve a task.
First transcript:
{transcript_1.to_str(transcript_idx_label=0)}
Second transcript:
{transcript_2.to_str(transcript_idx_label=1)}

We care about differences where the two agents have the same goal and context but take different actions. If the two agents have different goals or different context, that doesn't count, because differences in behavior are likely caused by upstream differences.
For instance, if both agents are testing code but the two pieces of code look very different, differences in the tests are not relevant (but the upstream difference in how the agents generated the code is worth looking into).

Here are some examples of differences, and the level of specifity in which we'd like them to be described:
- Both agents have read the task description and are trying to locate the test.py file, but agent 1 uses grep and succeeds while agent 2 uses tool calls and fails
- Both models encounter the same OutOfBoundsError when running their solutions and need to fix it, but agent 1 fixes it by modifying the test environment while agent 2 gives up
- Both models write similar solutions and are trying to test them, but agent 1 writes detailed tests while agent 2 does not

Follow the following procedure:
- For each assistant message in each transcript, write out the assistant's current state + the assistant's current goal + the action taken in the message. 
- Then, look through your summaries for all pairs of locations where the two agents have similar state + goals but take different actions, and list those.

Format your final list as follows:
Claim: both agents have similar context W and want to accomplish X, but agent 1 does Y while agent 2 does Z
Evidence: explain W, X, Y, Z, with citations
Claim: both agents have similar context W' and want to accomplish X', but agent 1 does Y' while agent 2 does Z'
Evidence: explain W', X', Y', Z', with citations
...

Avoid repeating yourself in the final list and avoid mentioning topics extremely similar to previously mentioned topics.
Always refer to the first transcript as "Agent 1" and the second as "Agent 2".
You are encouraged to cite evidence from the transcripts: {MULTI_BLOCK_CITE_INSTRUCTION}.
Avoid mentioning actions that both agents took, since that can never count as evidence for the two agents being different.
In the final list, explicitly mention the shared context and shared goals of the agents, and the different actions they took. Do not re-explain individual transcripts.


    """.strip()

    result = ""
    async def _streaming_callback(batch_index: int, llm_output: LLMOutput):
        nonlocal result

        result = llm_output.completions[0].text

    outputs = await get_llm_completions_async(
        [
            [
                {
                    "role": "user",
                    "content": prompt,
                },
            ]
        ],
        PROVIDER_PREFERENCES.compare_transcripts[0:1],
        max_new_tokens=8192 * 4,
        timeout=240.0,
        use_cache=True,
        streaming_callback=_streaming_callback,
    )

    text = outputs[0].first_text
    if text is None:
        return ""
    return text


from tqdm.asyncio import tqdm
diffs: dict[str, str] = {}
all_sample_ids = list(TRANSCRIPT_PAIRS.keys())
all_sample_ids = [
    "astropy__astropy-14096", "django__django-11551", "django__django-11239", "django__django-14351",
    "django__django-11299", "django__django-11749", "django__django-12193", "django__django-14404",
]
tasks = [
    compare_transcripts_3(TRANSCRIPT_PAIRS[sample_id][0], TRANSCRIPT_PAIRS[sample_id][1])
    for sample_id in all_sample_ids
]
results = await tqdm.gather(*tasks)
for i, result in enumerate(results):
    diffs[all_sample_ids[i]] = result
from IPython.display import clear_output
clear_output()
for k in diffs:
    print(k, TRANSCRIPT_PAIRS[k][0].metadata.scores, TRANSCRIPT_PAIRS[k][1].metadata.scores)
    print(diffs[k])
    print('-' * 100)

In [ ]:
# single pass, inline state reasoning with better instructions

async def compare_transcripts_4(
    transcript_1: Transcript,
    transcript_2: Transcript,
) -> str:
    prompt = f"""
Here are two different sequences of actions an agent took to solve a task.
First transcript:
{transcript_1.to_str(transcript_idx_label=0)}
Second transcript:
{transcript_2.to_str(transcript_idx_label=1)}

Note that each message in each sequence can have one of several roles - system, user, assistant, or tool.

For each ASSISTANT message in the first transcript, perform the following procedure:
- Summarize the action taken in the message
- Summarize the goal of the agent's current action
- Provide a concise but specific summary of the agent's past actions that are relevant to the current goal. You are encouraged to cite evidence from the transcripts: {MULTI_BLOCK_CITE_INSTRUCTION}

Do not mention non-assistant messages in your output.

Here are some examples of the level of specifity in which we'd like to describe the messages in.

Action: The agent uses grep to find the test.py file.
Goal: The agent is trying to explore the codebase and read code relevant to the task.capitalize
Relevant past actions: None, this is the first action taken by the agent.

Action: The agent is editing OAuth configuration settings in its test script.
Goal: The agent is trying to get its test script to run without errors.
Relevant past actions: The agent previously wrote a test script, but upon execution it produced an OutOfBoundsError. <further explanation and citations>

Action: The agent is writing a detailed test script that tests for many edge cases.
Goal: The agent is trying to test its solution to ensure correctness.
Relevant past actions: The agent previously implemented a solution that resolves the issue by modifying the function to use sets instead of lists. <further explanation and citations>

Then, do the same for the second transcript.

Format your output as follows:
[Message block citation]
Action: [action taken]
Goal: [goal of the action]
Relevant past actions: [summary of past actions that are relevant to the current goal, with citations]
[Message block citation]
Action: [action taken]
Goal: [goal of the action]
Relevant past actions: [summary of past actions that are relevant to the current goal, with citations]
...

--------------------------------

After you are done, proceed to the following step:

We care about differences where the two agents have the same goal and context but take different actions. If the two agents have different goals or different context, that doesn't count, because differences in behavior are likely caused by upstream differences.
For instance, if both agents are testing code but the two pieces of code look very different, differences in the tests are not relevant (but the upstream difference in how the agents generated the code is worth looking into).

Here are some examples of differences, and the level of specifity in which we'd like them to be described:
- Both agents have read the task description and are trying to locate the test.py file, but agent 1 uses grep and succeeds while agent 2 uses tool calls and fails
- Both models encounter the same OutOfBoundsError when running their solutions and need to fix it, but agent 1 fixes it by modifying the test environment while agent 2 gives up
- Both models write similar solutions and are trying to test them, but agent 1 writes detailed tests while agent 2 does not

Follow the following procedure:
- Look through your step 1 results for all pairs of locations where the two agents have similar state + goals but take different actions, and list those.

Format your final list as follows:
Claim: both agents have similar context W and want to accomplish X, but agent 1 does Y while agent 2 does Z
Evidence: explain W, X, Y, Z, with citations
Claim: both agents have similar context W' and want to accomplish X', but agent 1 does Y' while agent 2 does Z'
Evidence: explain W', X', Y', Z', with citations
...

Always refer to the first transcript as "Agent 1" and the second as "Agent 2".
Avoid mentioning actions that both agents took, since that can never count as evidence for the two agents being different.
In the final list, explicitly mention the shared context and shared goals of the agents, and the different actions they took.
Do not simply say the agents did something "differently"; instead explain how the two actions were different.


    """.strip()

    result = ""

    async def _streaming_callback(batch_index: int, llm_output: LLMOutput):
        nonlocal result

        result = llm_output.completions[0].text

    outputs = await get_llm_completions_async(
        [
            [
                {
                    "role": "user",
                    "content": prompt,
                },
            ]
        ],
        PROVIDER_PREFERENCES.compare_transcripts[0:1],
        max_new_tokens=8192 * 5,
        timeout=240.0,
        use_cache=True,
        streaming_callback=_streaming_callback,
    )

    text = outputs[0].first_text
    if text is None:
        return ""
    return text


from tqdm.asyncio import tqdm
diffs: dict[str, str] = {}
all_sample_ids = list(TRANSCRIPT_PAIRS.keys())
all_sample_ids = [
    "astropy__astropy-14096", "django__django-11551", "django__django-11239", "django__django-14351",
    "django__django-11299", "django__django-11749", "django__django-12193", "django__django-14404",
]
tasks = [
    compare_transcripts_4(TRANSCRIPT_PAIRS[sample_id][0], TRANSCRIPT_PAIRS[sample_id][1])
    for sample_id in all_sample_ids
]
results = await tqdm.gather(*tasks)
for i, result in enumerate(results):
    diffs[all_sample_ids[i]] = result
from IPython.display import clear_output
clear_output()
for k in diffs:
    print(k, TRANSCRIPT_PAIRS[k][0].metadata.scores, TRANSCRIPT_PAIRS[k][1].metadata.scores)
    print(diffs[k])
    print('-' * 100)

In [ ]:
# naive prompt, more specific controls

async def compare_transcripts_2(
    transcript_1: Transcript,
    transcript_2: Transcript,
) -> str:
    prompt = f"""
Here are two different sequences of actions an agent took to solve a task.
First transcript:
{transcript_1.to_str(transcript_idx_label=0)}
Second transcript:
{transcript_2.to_str(transcript_idx_label=1)}
Provide a list of key differences between the two transcripts. Do not re-explain individual transcripts.

We care about differences where the two agents have the same goal and context but take different actions. If the two agents have different goals or different context, that doesn't count, because differences in behavior are likely caused by upstream differences.
For instance, if both agents are testing code but the two pieces of code look very different, differences in the tests are not relevant (but the upstream difference in how the agents generated the code is worth looking into).
Avoid repeating yourself in the output and avoid mentioning topics extremely similar to previously mentioned topics.
Always refer to the first transcript as "Agent 1" and the second as "Agent 2".
You are encouraged to cite evidence from the transcripts: {MULTI_BLOCK_CITE_INSTRUCTION}.
Avoid mentioning actions that both agents took, since that can never count as evidence for the two agents being different.
Explicitly mention the shared context and shared goals of the agents, and the specific actions they took.

Here are some examples of differences, and the level of specifity in which we'd like them to be described:
- Both agents have read the task description and are trying to locate the test.py file, but agent 1 uses grep and succeeds while agent 2 uses tool calls and fails
- Both models encounter the same OutOfBoundsError when running their solutions and need to fix it, but agent 1 fixes it by modifying the test environment while agent 2 gives up
- Both models write similar solutions and are trying to test them, but agent 1 writes detailed tests while agent 2 does not

Format your output as follows:
Claim: both agents have similar context W and want to accomplish X, but agent 1 does Y while agent 2 does Z
Evidence: explain W, X, Y, Z, with citations
Claim: both agents have similar context W' and want to accomplish X', but agent 1 does Y' while agent 2 does Z'
Evidence: explain W', X', Y', Z', with citations
...
    """.strip()

    outputs = await get_llm_completions_async(
        [
            [
                {
                    "role": "user",
                    "content": prompt,
                },
            ]
        ],
        PROVIDER_PREFERENCES.compare_transcripts,
        max_new_tokens=8192 * 2,
        timeout=180.0,
        use_cache=True,
    )

    text = outputs[0].first_text
    if text is None:
        return ""
    return text


from tqdm.asyncio import tqdm
diffs: dict[str, str] = {}
all_sample_ids = list(TRANSCRIPT_PAIRS.keys())
all_sample_ids = [
    "astropy__astropy-14096", "django__django-11551", "django__django-11239", "django__django-14351",
    "django__django-11299", "django__django-11749", "django__django-12193", "django__django-14404",
]
tasks = [
    compare_transcripts_2(TRANSCRIPT_PAIRS[sample_id][0], TRANSCRIPT_PAIRS[sample_id][1])
    for sample_id in all_sample_ids
]
results = await tqdm.gather(*tasks)
for i, result in enumerate(results):
    diffs[all_sample_ids[i]] = result
from IPython.display import clear_output
clear_output()
for k in diffs:
    print(k, TRANSCRIPT_PAIRS[k][0].metadata.scores, TRANSCRIPT_PAIRS[k][1].metadata.scores)
    print(diffs[k])
    print('-' * 100)